# GPU timing of the full JAX model chain (release, DOP853 / Yoshida spray, score with 8 and 64 bootstraps)

Inputs: `pipe_inputs.tar` (uploaded in parts) in `My Drive/oCen_gpu/`. Code: `ocen_stream.jaxpipebench` from GitHub. Results: `My Drive/oCen_gpu/pipe_<gpu>/timing.json` (restartable per test point).

Runtime: **A100 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q --no-deps --force-reinstall "git+https://github.com/vasilybelokurov/ocen_stream@main"
!pip install -q diffrax
!git ls-remote https://github.com/vasilybelokurov/ocen_stream refs/heads/main
import jax, diffrax
print('jax', jax.__version__, 'diffrax', diffrax.__version__, jax.devices())
assert jax.devices()[0].platform == 'gpu', 'no GPU: change the runtime type'

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # a plain first mount fails after the permission pop-up
import os, glob, hashlib, shutil, tarfile
D = '/content/drive/MyDrive/oCen_gpu'
TAR = '/content/pipe_inputs.tar'
with open(TAR, 'wb') as fo:
    for f in sorted(glob.glob(D+'/pipe_inputs.tar.part_*')):
        with open(f, 'rb') as fi: shutil.copyfileobj(fi, fo)
sha = hashlib.sha256(open(TAR, 'rb').read()).hexdigest()
assert sha == '227c812f7b1046b7c91db71c9ebfb4efecd5a8d851d753ae1d2e43e8e2714a8f', sha
tarfile.open(TAR).extractall('/content/pipe_inputs')
GPU = jax.devices()[0].device_kind.replace(' ', '_')
OUT = f'{D}/pipe_{GPU}'
print(os.listdir('/content/pipe_inputs'), '->', OUT)

In [ ]:
!python -m ocen_stream.jaxpipebench --inputs /content/pipe_inputs --out "$OUT"